RAG Pipeline - Data ingestion to vector db

In [18]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [19]:
#Read all the pdf's in a directory

def process_all_pdfs(pdf_directory):
    """Process all the pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            #add source metadata to each document
            for doc in documents:
                doc.metadata["source"] = str(pdf_file)  
                doc.metadata["fileType"] = "pdf"
            
            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"Processed {len(all_documents)} documents from {len(pdf_files)} PDF files.")
    return all_documents

#Process all pdf's in the data/pdf directory
pdf_directory = "../data/pdf"
all_documents = process_all_pdfs(pdf_directory)

Found 2 PDF files in ../data/pdf
Processing ../data/pdf/Python Programming.pdf
Processing ../data/pdf/Word and Document Embeddings.pdf
Processed 237 documents from 2 PDF files.


In [20]:
all_documents

[Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-06-12T10:51:46+00:00', 'source': '../data/pdf/Python Programming.pdf', 'file_path': '../data/pdf/Python Programming.pdf', 'total_pages': 143, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-12T10:51:46+00:00', 'trapped': '', 'modDate': 'D:20260612105146Z', 'creationDate': 'D:20260612105146Z', 'page': 0, 'fileType': 'pdf'}, page_content='Python \nProgramming\nHans-Petter Halvorsen\nhttps://www.halvorsen.blog'),
 Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-06-12T10:51:46+00:00', 'source': '../data/pdf/Python Programming.pdf', 'file_path': '../data/pdf/Python Programming.pdf', 'total_pages': 143, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-12T10:51:46+00:00', 'trapped': '', 'modDate': 'D:20260612105146Z', 'creationDate': 'D:20260612105146Z', 'page': 1,

In [21]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    #show example of a chunk
    if split_docs:
        print(f"Example chunk: {split_docs[0].page_content[:500]}...")  # Print first 500 characters of the first chunk
        print(f"Metadata of the first chunk: {split_docs[0].metadata}")
        
    return split_docs

In [22]:
chunks = split_documents(all_documents)
chunks

Split 237 documents into 304 chunks.
Example chunk: Python 
Programming
Hans-Petter Halvorsen
https://www.halvorsen.blog...
Metadata of the first chunk: {'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-06-12T10:51:46+00:00', 'source': '../data/pdf/Python Programming.pdf', 'file_path': '../data/pdf/Python Programming.pdf', 'total_pages': 143, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-12T10:51:46+00:00', 'trapped': '', 'modDate': 'D:20260612105146Z', 'creationDate': 'D:20260612105146Z', 'page': 0, 'fileType': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-06-12T10:51:46+00:00', 'source': '../data/pdf/Python Programming.pdf', 'file_path': '../data/pdf/Python Programming.pdf', 'total_pages': 143, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-12T10:51:46+00:00', 'trapped': '', 'modDate': 'D:20260612105146Z', 'creationDate': 'D:20260612105146Z', 'page': 0, 'fileType': 'pdf'}, page_content='Python \nProgramming\nHans-Petter Halvorsen\nhttps://www.halvorsen.blog'),
 Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'TeX', 'creationdate': '2026-06-12T10:51:46+00:00', 'source': '../data/pdf/Python Programming.pdf', 'file_path': '../data/pdf/Python Programming.pdf', 'total_pages': 143, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-12T10:51:46+00:00', 'trapped': '', 'modDate': 'D:20260612105146Z', 'creationDate': 'D:20260612105146Z', 'page': 1,

###Embedding and Vector Store DB

In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully.")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        try:
            print(f"Generating embeddings for {len(texts)} texts.")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings of shape: {embeddings.shape}")
            print(f"Embedding dimension: {embeddings.shape[1]}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

##Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7600.92it/s]


Model all-MiniLM-L6-v2 loaded successfully.


##Vectorstore

In [25]:
class VectorStore:
    """Manage document embeddings in chromadb vector store."""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 304


In [26]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 304 texts.


Batches: 100%|██████████| 10/10 [00:01<00:00,  7.12it/s]

Generated embeddings of shape: (304, 384)
Embedding dimension: 384
Adding 304 documents to vector store...
Successfully added 304 documents to vector store
Total documents in collection: 608


In [27]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [28]:
rag_retriever

In [29]:
rag_retriever.retrieve("What is Python?")

Retrieving documents for query: 'What is Python?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 142.80it/s]


Generated embeddings of shape: (1, 384)
Embedding dimension: 384
Retrieved 5 documents (after filtering)


[{'id': 'doc_eeb9a451_1',
  'content': 'Python Programming',
  'metadata': {'creationdate': '2026-06-12T10:51:46+00:00',
   'author': '',
   'moddate': '2026-06-12T10:51:46+00:00',
   'page': 1,
   'fileType': 'pdf',
   'doc_index': 1,
   'producer': 'pdfTeX-1.40.28',
   'content_length': 18,
   'title': '',
   'modDate': 'D:20260612105146Z',
   'subject': '',
   'keywords': '',
   'file_path': '../data/pdf/Python Programming.pdf',
   'creator': 'TeX',
   'format': 'PDF 1.7',
   'creationDate': 'D:20260612105146Z',
   'total_pages': 143,
   'source': '../data/pdf/Python Programming.pdf',
   'trapped': ''},
  'similarity_score': 0.5663047432899475,
  'distance': 0.4336952567100525,
  'rank': 1},
 {'id': 'doc_261214b0_209',
  'content': 'Python Programming',
  'metadata': {'creationDate': 'D:20260612105146Z',
   'producer': 'pdfTeX-1.40.28',
   'fileType': 'pdf',
   'modDate': 'D:20260612105146Z',
   'title': '',
   'trapped': '',
   'subject': '',
   'doc_index': 209,
   'creationdate':

In [30]:
rag_retriever.retrieve("What is a neural network?")

Retrieving documents for query: 'What is a neural network?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 156.77it/s]

Generated embeddings of shape: (1, 384)
Embedding dimension: 384
Retrieved 0 documents (after filtering)


[]

RAG Pipeline- VectorDB To LLM Output Generation


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)  # Load environment variables from .env file, override existing ones

print(os.getenv("GROQ_API_KEY"))

In [60]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [64]:
class GroqLLM:
    def __init__(self, model_name: str = "qwen/qwen3.6-27b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [65]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: qwen/qwen3.6-27b
Groq LLM initialized successfully!


In [66]:
### get the context from the retriever and pass it to the LLM

context = rag_retriever.retrieve("What is Python?")
if context and groq_llm:
    response = groq_llm.generate_response("What is Python?", context)
    print("LLM Response:", response)

Retrieving documents for query: 'What is Python?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.78it/s]


Generated embeddings of shape: (1, 384)
Embedding dimension: 384
Retrieved 5 documents (after filtering)
LLM Response: 
<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "What is Python?"
   - **Context:** Provided as a list of dictionaries containing document chunks. Most chunks just say "Python Programming" with metadata. One chunk (doc_a2c1cd1b_34) contains actual text about Python: "Chapter 2 What is Python? 2.1 Introduction to Python Python is an open source and cross-platform programming language, that has become increasingly popular over the last ten years. It was first released in 1991. Latest version is 3.7.0. CPython is the reference implementation of the Python programming language. Written in C, CPython is the default and most widely-used implementation of the language. Python is a multi-purpose programming languages (due to its many extensions), examples are scientific computing and calculations, simulations, web development (using, e.g., t

Integration Vectordb Context pipeline With LLM output


In [67]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3.6-27b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [68]:
answer=rag_simple("What is Python?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Python?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.30it/s]


Generated embeddings of shape: (1, 384)
Embedding dimension: 384
Retrieved 3 documents (after filtering)

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** "Python Programming\n\nPython Programming\n\nPython Programming"
   - **Question:** "What is Python?"
   - **Instruction:** "Use the following context to answer the question concisely."

2.  **Evaluate Context:**
   - The provided context is just the repeated phrase "Python Programming". It contains no actual information about what Python is.

3.  **Identify Constraint/Conflict:**
   - The instruction says to use the context to answer concisely.
   - The context is insufficient to answer the question.
   - I need to acknowledge the limitation of the context while still providing a concise, accurate answer based on general knowledge, or strictly state that the context doesn't provide the answer. However, standard AI behavior for this type of prompt is to note the context's limitation but still answer t

Enhanced RAG Pipeline Features

In [70]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("The new age of programming", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'The new age of programming'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


Generated embeddings of shape: (1, 384)
Embedding dimension: 384
Retrieved 2 documents (after filtering)
Answer: 
<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** A repeated excerpt from a textbook introduction about Python programming. It discusses "The New Age of Programming," noting changes over the last 30 years (from early 80s PCs to modern smartphones/tablets/PCs), the impact of the Internet, the dominance of web sites/apps over traditional desktop apps, and the need to learn suitable programming languages for this new age.
   - **Question:** "The new age of programming"
   - **Task:** Answer concisely based on the context.

2.  **Identify Key Information in Context related to the Question:**
   - The context defines "The New Age of Programming" as a period marked by dramatic changes in software creation over the last 30 years.
   - It highlights the shift from early personal computers to powerful modern devices (smartphones, tablets, PCs).
   - I

In [72]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Python?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is Python?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.28it/s]

Generated embeddings of shape: (1, 384)
Embedding dimension: 384
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Python Prog

ramming

Python Programming

Python Programming

Question: what is Python?

Answer:

Final Answer: 
<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** "Python Programming\n\nPython Programming\n\nPython Programming"
   - **Question:** "what is Python?"
   - **Instruction:** "Use the following context to answer the question concisely."

2.  **Evaluate Context:**
   - The provided context is just the repeated phrase "Python Programming". It contains no actual information about what Python is.

3.  **Identify Constraint/Challenge:**
   - The instruction says "Use the following context to answer the question concisely."
   - However, the context is insufficient to answer the question.
   - I need to acknowledge the context limitation while still providing a concise, accurate answer based on general knowledge, or strictly follow the constraint if possible. Usually, in these tasks, if the context is empty/repetitive, I should state that the context doesn't prov